In [2]:
import sys
import os


sys.path.append('..')
os.chdir('..')

In [3]:
# dataset
import os
import torch

from starry.utils.config import Configuration
from starry.utils.dataset_factory import loadDataset
from starry.utils.model_factory import loadModel


torch.set_printoptions(profile="full")

DATA_DIR = os.getenv('DATA_DIR')

config = Configuration.create('./configs/paraff-visionund-test.yaml', volatile=True)
train, = loadDataset(config, data_dir=DATA_DIR, splits='*0/1', device='cuda')
model = loadModel(config['model'], postfix='Loss').cuda()

it = iter(train)
batch = next(it)
batch['input_ids']

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.
2025-03-17 18:45:12.743224: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-17 18:45:12.743260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-17 18:45:12

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

tensor([[100000,   2054,    418,    245,   9394,   4706,    285,  10046,  20308,
             13,   1257,    418,   2249,    276,   2579,    254,   7959,   3093,
            344,    254,   2677,   4614,     11,    285,   4750,    254,   2677,
            366,    245,   6265,    280,   9224,   1244,   3892,   4706,     13,
            185,    185, 100601,     25,    207, 100016, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100

In [ ]:
prompt_len = batch['target_mask'][0].tolist().index(True)
prompt_len

631

In [14]:
input_ids = batch['input_ids'][:, :prompt_len]
input_ids

tensor([[100000,   2054,    418,    245,   9394,   4706,    285,  10046,  20308,
             13,   1257,    418,   2249,    276,   2579,    254,   7959,   3093,
            344,    254,   2677,   4614,     11,    285,   4750,    254,   2677,
            366,    245,   6265,    280,   9224,   1244,   3892,   4706,     13,
            185,    185, 100601,     25,    207, 100016, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594, 100594,
         100594, 100594, 100

In [15]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/Janus-Pro-7B")

input_text = tokenizer.decode(input_ids[0])
input_text

'<｜begin▁of▁sentence｜>You are a helpful language and vision assistant. You are able to understand the visual content that the user provides, and assist the user with a variety of tasks using natural language.\n\n<|User|>: <begin_of_image><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><image_placeholder><i

In [16]:
img_emb = model.aligner(batch['img_emb'].to(model.dtype))
img_emb.shape

torch.Size([1, 1, 576, 4096])

In [18]:
inputs_embeds = model.deducer.get_input_embeddings()(input_ids)
image_seq_mask = batch['image_seq_mask']
img_emb = img_emb.reshape((-1, img_emb.shape[-1]))
inputs_embeds[image_seq_mask[:, :prompt_len]] = img_emb

inputs_embeds.shape

torch.Size([1, 631, 4096])

In [20]:
outputs = model.deducer.generate(
    inputs_embeds=inputs_embeds,
    attention_mask=batch['attention_mask'][:, :prompt_len],
    pad_token_id=tokenizer.eos_token_id,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=64,
    do_sample=False,
    use_cache=True,
    temperature=0,
    top_p=1,
)

outputs

tensor([[    40,      6,     76,  11547,     11,    548,    304,   2977,  13341,
            254,   8121,   1244,   3126,   3555,     13,   3126,   3555,    317,
            245,   5278,    327,  32773,    285,   6714,   9686,  15385,     11,
            548,    359,   1217,    441,    463,    254,  19952,    276,  13341,
            410,   6778,   9686,  15385,   4723,     13, 100001]],
       device='cuda:0')

In [21]:
tokenizer.decode(outputs[0])

"I'm sorry, but I cannot recognize the score using Paraff. Paraff is a tool for analyzing and understanding musical scores, but it does not have the capability to recognize or interpret musical scores directly.<｜end▁of▁sentence｜>"